# 02 — Data Preparation
### Track B — Customer Analytics cho E-Commerce | IT701220 Data Mining (Cao học)
**Người phụ trách:** Thành viên A — Data & Infrastructure Lead

---

**Input:** `data/raw/online_retail_II.xlsx` (đọc lại từ đầu, không phụ thuộc notebook 01)

**Output — 3 file bàn giao cho cả nhóm:**
| File | Nơi lưu | Người đọc | Vai trò |
|---|---|---|---|
| `transactions_clean.csv` | `data/processed/` | Thành viên B (MBA) | Giữ dòng thiếu Customer ID |
| `customer_rfm.csv` | `data/processed/` | Thành viên C (Clustering) | RFM tính TOÀN KỲ |
| `customer_churn_features.csv` | `data/processed/` | Thành viên D (Churn) | RFM tính CẮT TẠI SNAPSHOT |

**Notebook này LÀM GÌ:** gọi lại các hàm đã viết ở `src/preprocessing/` và `src/rfm/` (đã test kỹ ở
bước trước), áp dụng lên dữ liệu thô, xuất ra đúng 3 file trên theo đúng cấu trúc cột đã thống nhất
trong bảng "Checklist bàn giao" của cả nhóm.

**Notebook này KHÔNG làm gì:** không tự ý chạy Apriori/K-Means/Decision Tree — các bước đó thuộc phạm
vi của Thành viên B/C/D ở notebook 03/04/05.

---

### ⚠️ LƯU Ý QUAN TRỌNG VỀ MỐC SNAPSHOT CHO CHURN (đọc kỹ trước khi dùng số liệu ở đây)

Theo đúng quy trình đã thống nhất trong tài liệu phân công: **mốc `snapshot_date` dùng để tính
`customer_churn_features.csv` KHÔNG được Thành viên A tự quyết định một mình** — phải chờ Thành viên D
phân tích phân phối *inter-purchase gap* (khoảng cách giữa các lần mua liên tiếp) rồi đề xuất con số có
căn cứ thống kê.

Ở notebook này, mình tạm tính sẵn phân phối inter-purchase gap (dùng chung công thức D sẽ dùng trong
`src/churn/label_generator.py`) để có MỘT MỐC PLACEHOLDER hợp lý, giúp pipeline chạy được ngay từ
Midterm. **Con số `SNAPSHOT_CHURN` ở Mục 3 dưới đây CẦN được A trao đổi lại với D và cập nhật khi D đã
chốt xong phân tích chính thức của mình** — đây không phải quyết định cuối cùng.


## 0. Chuẩn bị môi trường

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Thêm thư mục gốc dự án vào sys.path để import được module trong src/
thu_muc_hien_tai = Path.cwd()
if (thu_muc_hien_tai / "src").exists():
    PROJECT_ROOT_TAM = thu_muc_hien_tai
else:
    PROJECT_ROOT_TAM = thu_muc_hien_tai.parent
sys.path.insert(0, str(PROJECT_ROOT_TAM))

# Import các hàm đã viết và TEST KỸ ở các bước trước
from src.utils.io import load_raw_excel, save_to_csv, get_project_root, PROCESSED_DATA_DIR
from src.preprocessing.cleaning import clean_transactions
from src.preprocessing.feature_engineering import add_total_price_column
from src.rfm.rfm_calculator import calculate_rfm

print(f"Thư mục gốc dự án: {get_project_root()}")
print(f"Thư mục xuất file : {PROCESSED_DATA_DIR}")


Thư mục gốc dự án: /home/claude/project
Thư mục xuất file : /home/claude/project/data/processed


## 1. Đọc dữ liệu thô + chạy pipeline làm sạch

Gọi lại `load_raw_excel()` (đọc + gộp 2 sheet) và `clean_transactions()` (pipeline làm sạch tổng hợp đã
viết và test ở `src/preprocessing/cleaning.py`). Toàn bộ log chi tiết từng bước sẽ được in ra tự động
bởi các hàm này.

In [ ]:
df_tho = load_raw_excel()
print(f"Dữ liệu thô: {df_tho.shape[0]:,} dòng × {df_tho.shape[1]} cột\n")

df_sach = clean_transactions(df_tho)


Dữ liệu thô: 1,067,371 dòng × 9 cột

BẮT ĐẦU PIPELINE LÀM SẠCH DỮ LIỆU
Số dòng ban đầu: 1,067,371
----------------------------------------------------------------------


[remove_cancelled_invoices] Đã loại 19,494 dòng invoice huỷ (còn lại 1,047,877 dòng).


[remove_bad_debt_adjustments] Đã loại 6 dòng 'Adjust bad debt' (còn lại 1,047,871 dòng).


[remove_stock_adjustment_rows] Đã loại 3,457 dòng Quantity âm bất thường (còn lại 1,044,414 dòng).


[remove_non_product_stock_codes] Đã loại 4,613 dòng StockCode phi sản phẩm (còn lại 1,039,801 dòng).


[remove_duplicate_rows] Đã loại 33,788 dòng trùng lặp (còn lại 1,006,013 dòng).
----------------------------------------------------------------------
Số dòng sau khi làm sạch: 1,006,013 (đã loại tổng cộng 61,358 dòng, tương đương 5.75%)


In [ ]:
# Tính thêm cột TotalPrice = Quantity * Price - dùng chung cho cả MBA (tham khảo)
# và bắt buộc cho RFM/Churn (thành phần tính Monetary)
df_sach = add_total_price_column(df_sach)

df_sach[["Quantity", "Price", "TotalPrice"]].describe()


,Quantity,Price,TotalPrice
count,1.006013e+06,1.006013e+06,1.006013e+06
mean,1.136363e+01,3.333701e+00,1.952892e+01
std,1.317178e+02,4.778390e+00,1.997205e+02
min,1.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,1.250000e+00,3.950000e+00
50%,4.000000e+00,2.100000e+00,1.000000e+01
75%,1.200000e+01,4.130000e+00,1.770000e+01
max,8.099500e+04,1.157150e+03,1.684696e+05


## 2. Xuất File 1 — `transactions_clean.csv` (cho Thành viên B — MBA)

**Quy tắc quan trọng:** file này GIỮ NGUYÊN các dòng thiếu `Customer ID` — vì luật kết hợp (Association
Rules) chỉ cần biết "sản phẩm nào đi cùng sản phẩm nào" trong 1 hoá đơn, KHÔNG cần biết ai là người mua.
Nếu loại bỏ các dòng này, MBA sẽ mất khoảng 22.8% dữ liệu giao dịch một cách không cần thiết.

In [ ]:
# KHÔNG loại dòng thiếu Customer ID ở đây - đây chính xác là input cho MBA
transactions_clean = df_sach.copy()

# Loại cột phụ "Sheet" (chỉ dùng nội bộ để kiểm tra duplicate ở notebook 01),
# không cần thiết cho các bước phân tích tiếp theo
if "Sheet" in transactions_clean.columns:
    transactions_clean = transactions_clean.drop(columns=["Sheet"])

print(f"transactions_clean: {transactions_clean.shape[0]:,} dòng × {transactions_clean.shape[1]} cột")
print(f"Tỷ lệ vẫn còn thiếu Customer ID: {transactions_clean['Customer ID'].isnull().mean()*100:.2f}% "
      f"(đúng như dự kiến - KHÔNG loại bỏ cho pipeline MBA)")

save_to_csv(transactions_clean, PROCESSED_DATA_DIR / "transactions_clean.csv")


transactions_clean: 1,006,013 dòng × 9 cột
Tỷ lệ vẫn còn thiếu Customer ID: 22.80% (đúng như dự kiến - KHÔNG loại bỏ cho pipeline MBA)


Đã lưu file: /home/claude/project/data/processed/transactions_clean.csv  (số dòng: 1,006,013 | số cột: 9)


In [ ]:
# Tự kiểm tra lại: đọc lại file vừa xuất để chắc chắn không lỗi format
# (bước kiểm tra nhanh mà bảng "Checklist bàn giao" của nhóm yêu cầu)
kiem_tra_lai = pd.read_csv(PROCESSED_DATA_DIR / "transactions_clean.csv")
print("Đọc lại file vừa xuất - OK.")
print(f"Shape: {kiem_tra_lai.shape}")
print(f"Các cột: {list(kiem_tra_lai.columns)}")
kiem_tra_lai.head(3)


Đọc lại file vừa xuất - OK.
Shape: (1006013, 9)
Các cột: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'TotalPrice']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0


## 3. Xác định mốc Snapshot cho Churn (phối hợp với Thành viên D)

Trước khi xuất `customer_rfm.csv` và `customer_churn_features.csv`, cần xác định 2 mốc `snapshot_date`
khác nhau:

- **Snapshot cho C (RFM/Clustering):** đơn giản — dùng ngày giao dịch cuối cùng trong dữ liệu (toàn kỳ).
- **Snapshot cho D (Churn):** phức tạp hơn — cần dựa trên phân phối *inter-purchase gap* để chọn có căn
  cứ thống kê, KHÔNG chọn số tuỳ tiện (ví dụ không tự ý chọn "90 ngày" mà không giải thích được).

Phần dưới đây tính trước phân phối inter-purchase gap (trên dữ liệu ĐÃ LÀM SẠCH, loại Customer ID
thiếu) để cả nhóm có căn cứ thảo luận — đây là bước phân tích chính thức thuộc về Thành viên D, ở đây
A chỉ tính trước 1 bản để có mốc placeholder cho pipeline chạy được.

In [ ]:
du_lieu_co_khach = df_sach[df_sach["Customer ID"].notna()].sort_values(
    ["Customer ID", "InvoiceDate"]
)

# Với mỗi khách hàng, lấy danh sách CÁC NGÀY MUA DUY NHẤT (unique), sắp theo thời gian
ngay_mua_theo_khach = du_lieu_co_khach.groupby("Customer ID")["InvoiceDate"].apply(
    lambda x: sorted(x.unique())
)

# Tính khoảng cách (tính bằng ngày) giữa 2 lần mua liên tiếp của CÙNG 1 khách hàng
danh_sach_khoang_cach = []
for cac_ngay in ngay_mua_theo_khach:
    if len(cac_ngay) > 1:
        chuoi_ngay = pd.to_datetime(pd.Series(cac_ngay))
        khoang_cach = chuoi_ngay.diff().dropna().dt.days
        danh_sach_khoang_cach.extend(khoang_cach.tolist())

khoang_cach_mua_hang = pd.Series(danh_sach_khoang_cach)

print(f"Số cặp lần-mua-liên-tiếp thu được: {len(khoang_cach_mua_hang):,} "
      f"(từ {ngay_mua_theo_khach.apply(len).gt(1).sum():,} khách có ≥ 2 lần mua khác ngày)")
print(f"\nSố khách CHỈ mua đúng 1 lần: {(ngay_mua_theo_khach.apply(len) == 1).sum():,} "
      f"/ {len(ngay_mua_theo_khach):,} "
      f"({(ngay_mua_theo_khach.apply(len) == 1).mean()*100:.1f}%) "
      f"- nhóm này KHÔNG có 'gap' để tính, Thành viên D cần xử lý riêng.")

print("\nPhân phối inter-purchase gap (ngày) theo percentile:")
for p in [50, 75, 80, 90, 95, 99]:
    print(f"  P{p}: {khoang_cach_mua_hang.quantile(p/100):.1f} ngày")


Số cặp lần-mua-liên-tiếp thu được: 30,632 (từ 4,235 khách có ≥ 2 lần mua khác ngày)

Số khách CHỈ mua đúng 1 lần: 1,618 / 5,853 (27.6%) - nhóm này KHÔNG có 'gap' để tính, Thành viên D cần xử lý riêng.

Phân phối inter-purchase gap (ngày) theo percentile:
  P50: 25.0 ngày
  P75: 62.0 ngày
  P80: 77.0 ngày
  P90: 136.0 ngày
  P95: 207.0 ngày
  P99: 369.0 ngày


**Nhận xét:** Trung vị khoảng cách giữa 2 lần mua liên tiếp khá ngắn, nhưng phân phối có đuôi dài
— một số khách hàng có khoảng cách giữa các lần mua lên tới hàng trăm ngày. Đây là cơ sở để Thành viên D
chọn cửa sổ churn — ví dụ dùng mốc quanh percentile 75-90 làm ngưỡng "không quay lại mua trong khoảng
thời gian này thì coi là có nguy cơ churn". Quyết định CHÍNH THỨC và LÝ GIẢI chi tiết thuộc trách nhiệm
của Thành viên D trong `src/churn/label_generator.py`.

In [ ]:
# ===== MỐC PLACEHOLDER - CẦN D XÁC NHẬN LẠI, KHÔNG PHẢI QUYẾT ĐỊNH CUỐI CÙNG =====
# Ở đây tạm dùng percentile 75 làm độ dài cửa sổ churn placeholder, để pipeline
# có số liệu chạy thử ngay từ Midterm. Thành viên D cần thay thế số này bằng
# phân tích chính thức của mình (có thể là percentile khác, hoặc kết hợp thêm
# tiêu chí nghiệp vụ khác) trước khi notebook 05 sinh nhãn churn thật.

DO_DAI_CUA_SO_CHURN_PLACEHOLDER = int(khoang_cach_mua_hang.quantile(0.75))
print(f"Độ dài cửa sổ churn placeholder (percentile 75): "
      f"{DO_DAI_CUA_SO_CHURN_PLACEHOLDER} ngày")
print("=> CẦN TRAO ĐỔI VỚI THÀNH VIÊN D để chốt con số chính thức trước khi dùng cho Final.")

ngay_cuoi_du_lieu = df_sach["InvoiceDate"].max()
SNAPSHOT_TOAN_KY = ngay_cuoi_du_lieu  # dùng cho Thành viên C
SNAPSHOT_CHURN = ngay_cuoi_du_lieu - pd.Timedelta(days=DO_DAI_CUA_SO_CHURN_PLACEHOLDER)  # dùng cho Thành viên D

print(f"\nSNAPSHOT_TOAN_KY (cho C) = {SNAPSHOT_TOAN_KY}")
print(f"SNAPSHOT_CHURN   (cho D) = {SNAPSHOT_CHURN}  (placeholder, chờ D xác nhận)")


Độ dài cửa sổ churn placeholder (percentile 75): 62 ngày
=> CẦN TRAO ĐỔI VỚI THÀNH VIÊN D để chốt con số chính thức trước khi dùng cho Final.

SNAPSHOT_TOAN_KY (cho C) = 2011-12-09 12:50:00
SNAPSHOT_CHURN   (cho D) = 2011-10-08 12:50:00  (placeholder, chờ D xác nhận)


## 4. Xuất File 2 — `customer_rfm.csv` (cho Thành viên C — Clustering)

Gọi hàm `calculate_rfm()` (đã test kỹ ở bước trước) với `snapshot_date = SNAPSHOT_TOAN_KY` — tức dùng TOÀN
BỘ lịch sử giao dịch để mô tả khách hàng. Đây là input cho bước phân khúc khách hàng (Customer
Segmentation), không có khái niệm "dự đoán tương lai" nên dùng toàn bộ dữ liệu là hợp lệ.

In [ ]:
customer_rfm = calculate_rfm(df_sach, snapshot_date=SNAPSHOT_TOAN_KY)

print("\nThống kê RFM (toàn kỳ):")
customer_rfm.describe()


[calculate_rfm] Đã loại 229,371 dòng thiếu 'Customer ID' (còn lại 776,642 dòng để tính RFM).
[calculate_rfm] Mốc snapshot_date = 2011-12-09. Đã loại 0 dòng có ngày giao dịch SAU mốc này (còn lại 776,642 dòng).
  (Lưu ý: 0 dòng bị loại nghĩa là snapshot_date đang lớn hơn hoặc bằng ngày giao dịch cuối cùng trong dữ liệu - đúng như kỳ vọng khi tính RFM 'toàn kỳ' cho Thành viên C.)
[calculate_rfm] Đã tính RFM cho 5,853 khách hàng.

Thống kê RFM (toàn kỳ):


,Customer ID,Recency,Frequency,Monetary
count,5853.000000,5853.000000,5853.000000,5853.000000
mean,15319.354519,199.166240,6.253203,2916.335857
std,1714.995565,208.505959,12.751977,14305.788772
min,12346.000000,0.000000,1.000000,0.000000
25%,13837.000000,24.000000,1.000000,339.500000
50%,15320.000000,94.000000,3.000000,856.010000
75%,16802.000000,378.000000,7.000000,2240.900000
max,18287.000000,738.000000,373.000000,580987.040000


In [ ]:
save_to_csv(customer_rfm, PROCESSED_DATA_DIR / "customer_rfm.csv")

# Tự kiểm tra lại
kiem_tra_rfm = pd.read_csv(PROCESSED_DATA_DIR / "customer_rfm.csv")
print("Đọc lại file vừa xuất - OK.")
print(f"Shape: {kiem_tra_rfm.shape}")
print(f"Các cột: {list(kiem_tra_rfm.columns)}")
kiem_tra_rfm.head(3)


Đã lưu file: /home/claude/project/data/processed/customer_rfm.csv  (số dòng: 5,853 | số cột: 4)
Đọc lại file vừa xuất - OK.
Shape: (5853, 4)
Các cột: ['Customer ID', 'Recency', 'Frequency', 'Monetary']


,Customer ID,Recency,Frequency,Monetary
0,12346.0,325,3,77352.96
1,12347.0,1,8,4921.53
2,12348.0,74,5,1658.40


## 5. Xuất File 3 — `customer_churn_features.csv` (cho Thành viên D — Churn)

Gọi lại CÙNG hàm `calculate_rfm()` nhưng với `snapshot_date = SNAPSHOT_CHURN` (mốc cắt sớm hơn, hiện đang
là placeholder — xem cảnh báo ở Mục 3). Đây là điểm kỹ thuật QUAN TRỌNG NHẤT để chống rò rỉ dữ liệu:
các giao dịch xảy ra SAU mốc `SNAPSHOT_CHURN` sẽ KHÔNG được dùng để tính RFM ở đây, và sẽ được Thành
viên D dùng riêng ở bước sau (`label_generator.py`) để xác nhận khách có thực sự quay lại mua hay
không — tức là để sinh ra NHÃN churn (0/1).

In [ ]:
customer_churn_features = calculate_rfm(df_sach, snapshot_date=SNAPSHOT_CHURN)

print("\nThống kê RFM (cắt tại snapshot churn - placeholder):")
customer_churn_features.describe()


[calculate_rfm] Đã loại 229,371 dòng thiếu 'Customer ID' (còn lại 776,642 dòng để tính RFM).


[calculate_rfm] Mốc snapshot_date = 2011-10-08. Đã loại 116,778 dòng có ngày giao dịch SAU mốc này (còn lại 659,864 dòng).
[calculate_rfm] Đã tính RFM cho 5,472 khách hàng.

Thống kê RFM (cắt tại snapshot churn - placeholder):


,Customer ID,Recency,Frequency,Monetary
count,5472.000000,5472.000000,5472.000000,5472.000000
mean,15318.675621,203.426718,5.800804,2687.816343
std,1710.655732,186.697810,11.592128,12993.814994
min,12346.000000,0.000000,1.000000,2.900000
25%,13838.750000,32.000000,1.000000,321.125000
50%,15313.500000,144.000000,3.000000,785.030000
75%,16798.250000,339.000000,6.000000,2112.040000
max,18287.000000,676.000000,308.000000,524476.600000


In [ ]:
save_to_csv(customer_churn_features, PROCESSED_DATA_DIR / "customer_churn_features.csv")

# Tự kiểm tra lại
kiem_tra_churn = pd.read_csv(PROCESSED_DATA_DIR / "customer_churn_features.csv")
print("Đọc lại file vừa xuất - OK.")
print(f"Shape: {kiem_tra_churn.shape}")
print(f"Các cột: {list(kiem_tra_churn.columns)}")
kiem_tra_churn.head(3)


Đã lưu file: /home/claude/project/data/processed/customer_churn_features.csv  (số dòng: 5,472 | số cột: 4)
Đọc lại file vừa xuất - OK.
Shape: (5472, 4)
Các cột: ['Customer ID', 'Recency', 'Frequency', 'Monetary']


,Customer ID,Recency,Frequency,Monetary
0,12346.0,263,3,77352.96
1,12347.0,67,6,3402.39
2,12348.0,12,5,1658.40


**Lưu ý nhắc lại cho Thành viên D:** file `customer_churn_features.csv` này hiện tính theo mốc
snapshot PLACEHOLDER (percentile 75 của inter-purchase gap toàn dữ liệu). Khi D đã hoàn thành phân
tích chính thức trong `src/churn/label_generator.py` và chốt được mốc + độ dài cửa sổ churn cuối cùng
(có thể khác với placeholder này, ví dụ D quyết định dùng percentile khác, hoặc tính gap theo cách
khác), A cần CHẠY LẠI notebook này với giá trị `SNAPSHOT_CHURN` mới do D cung cấp, rồi xuất lại file.

## 6. So sánh nhanh 2 bảng RFM — minh hoạ trực quan sự khác biệt

In [ ]:
bang_so_sanh = pd.DataFrame({
    "Chỉ số": ["Số khách hàng", "Frequency trung bình", "Monetary trung bình (£)",
               "Recency trung bình (ngày)"],
    "customer_rfm.csv (toàn kỳ - cho C)": [
        len(customer_rfm),
        round(customer_rfm["Frequency"].mean(), 2),
        round(customer_rfm["Monetary"].mean(), 2),
        round(customer_rfm["Recency"].mean(), 2),
    ],
    "customer_churn_features.csv (cắt snapshot - cho D)": [
        len(customer_churn_features),
        round(customer_churn_features["Frequency"].mean(), 2),
        round(customer_churn_features["Monetary"].mean(), 2),
        round(customer_churn_features["Recency"].mean(), 2),
    ],
})
bang_so_sanh


,Chỉ số,customer_rfm.csv (toàn kỳ - cho C),customer_churn_features.csv (cắt snapshot - cho D)
0,Số khách hàng,5853.00,5472.00
1,Frequency trung bình,6.25,5.80
2,Monetary trung bình (£),2916.34,2687.82
3,Recency trung bình (ngày),199.17,203.43


**Nhận xét:** Bảng trên cho thấy rõ ràng 2 file có giá trị khác nhau (Frequency/Monetary ở bản
cắt snapshot thấp hơn bản toàn kỳ, vì bị cắt bớt dữ liệu "tương lai") — đây chính là bằng chứng số liệu
nhóm có thể đưa vào báo cáo để giải thích tại sao KHÔNG được dùng nhầm `customer_rfm.csv` để train mô
hình churn (nếu dùng nhầm, mô hình sẽ "nhìn thấy trước" thông tin lẽ ra chưa xảy ra tại thời điểm dự
đoán — lỗi rò rỉ dữ liệu nghiêm trọng).

## 7. Tổng kết bàn giao

| File | Trạng thái | Cột | Người nhận |
|---|---|---|---|
| `data/processed/transactions_clean.csv` | ✅ Đã xuất | Invoice, StockCode, Description, Quantity, Price, InvoiceDate, Customer ID, Country, TotalPrice | Thành viên B |
| `data/processed/customer_rfm.csv` | ✅ Đã xuất | Customer ID, Recency, Frequency, Monetary | Thành viên C |
| `data/processed/customer_churn_features.csv` | ⚠️ Đã xuất (mốc PLACEHOLDER — cần D xác nhận lại) | Customer ID, Recency, Frequency, Monetary | Thành viên D |

**Việc cần làm tiếp theo:**
1. Báo B, C có thể bắt đầu đọc file của mình ngay (không cần chờ gì thêm).
2. Trao đổi trực tiếp với D về mốc `SNAPSHOT_CHURN` — sau khi D chốt xong phân tích chính thức trong
   `src/churn/label_generator.py`, quay lại chạy lại Mục 3 và Mục 5 của notebook này với con số D cung
   cấp, xuất lại `customer_churn_features.csv`.
3. Restart & Run All lại toàn bộ notebook trước khi coi là hoàn thành, đảm bảo không có cell nào chạy
   lỗi thứ tự.